# 🏨 Детекция неполных описаний комнат для Т-Путешествия

## Задача
Предсказать, что в описании комнаты недостаточно данных для матчинга с мастер-комнатой, чтобы минимизировать поток заданий в краудсорсинг.

## Метрики
- **Precision (Точность)**: минимизация ложных срабатываний
- **Recall (Полнота)**: максимизация обнаружения неполных описаний
- **F1-score**: баланс между точностью и полнотой

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Загрузка и первичный анализ данных

In [ ]:
df = pd.read_csv('hotel_rooms_data.csv')
print(f"Размер датасета: {df.shape}")
print(f"\nРаспределение целевой переменной:")
print(df['is_incomplete'].value_counts())
print(f"\nДоля неполных описаний: {df['is_incomplete'].mean()*100:.1f}%")
df.head()

In [ ]:
df.info()

## 2. Feature Engineering

Создаем признаки, которые помогут определить полноту описания:
- Количество пропущенных значений
- Длина текстовых полей
- Наличие ключевых атрибутов

In [ ]:
def create_features(df):
    df = df.copy()
    
    # Количество пропущенных значений
    df['missing_count'] = df[['area', 'bed_type', 'view', 'max_guests', 'floor', 'n_rooms', 'rating']].isna().sum(axis=1)
    
    # Длина текстовых полей
    df['room_name_len'] = df['room_name'].fillna('').str.len()
    df['description_len'] = df['description'].fillna('').str.len()
    df['amenities_count'] = df['amenities'].fillna('').str.split(',').str.len()
    
    # Бинарные признаки наличия данных
    df['has_area'] = df['area'].notna().astype(int)
    df['has_bed_type'] = (df['bed_type'].fillna('') != '').astype(int)
    df['has_view'] = (df['view'].fillna('') != '').astype(int)
    df['has_max_guests'] = df['max_guests'].notna().astype(int)
    df['has_floor'] = df['floor'].notna().astype(int)
    df['has_n_rooms'] = df['n_rooms'].notna().astype(int)
    df['has_rating'] = df['rating'].notna().astype(int)
    df['has_description'] = (df['description_len'] > 0).astype(int)
    df['has_room_name'] = (df['room_name_len'] > 0).astype(int)
    
    # Заполнение пропусков
    df['area'] = df['area'].fillna(df['area'].median())
    df['max_guests'] = df['max_guests'].fillna(df['max_guests'].median())
    df['floor'] = df['floor'].fillna(df['floor'].median())
    df['n_rooms'] = df['n_rooms'].fillna(df['n_rooms'].median())
    df['rating'] = df['rating'].fillna(df['rating'].median())
    
    return df

df = create_features(df)
print("Признаки созданы")
df.head()

## 3. Визуализация данных

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Распределение пропущенных значений
df.groupby('is_incomplete')['missing_count'].mean().plot(kind='bar', ax=axes[0,0], color=['green', 'red'])
axes[0,0].set_title('Среднее количество пропусков')
axes[0,0].set_xlabel('Неполное описание')
axes[0,0].set_ylabel('Количество пропусков')

# Длина описания
df.groupby('is_incomplete')['description_len'].mean().plot(kind='bar', ax=axes[0,1], color=['green', 'red'])
axes[0,1].set_title('Средняя длина описания')
axes[0,1].set_xlabel('Неполное описание')

# Количество удобств
df.groupby('is_incomplete')['amenities_count'].mean().plot(kind='bar', ax=axes[1,0], color=['green', 'red'])
axes[1,0].set_title('Среднее количество удобств')
axes[1,0].set_xlabel('Неполное описание')

# Количество фото
df.groupby('is_incomplete')['n_photos'].mean().plot(kind='bar', ax=axes[1,1], color=['green', 'red'])
axes[1,1].set_title('Среднее количество фото')
axes[1,1].set_xlabel('Неполное описание')

plt.tight_layout()
plt.show()

## 4. Подготовка данных для обучения

In [ ]:
# Выбор признаков
feature_cols = [
    'area', 'max_guests', 'floor', 'n_rooms', 'price', 'rating', 'n_photos',
    'missing_count', 'room_name_len', 'description_len', 'amenities_count',
    'has_area', 'has_bed_type', 'has_view', 'has_max_guests', 'has_floor',
    'has_n_rooms', 'has_rating', 'has_description', 'has_room_name'
]

X = df[feature_cols]
y = df['is_incomplete']

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Обучение моделей

Тестируем несколько алгоритмов для выбора лучшего

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Обучение: {name}")
    print('='*50)
    
    # Обучение
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # Метрики
    print(classification_report(y_test, y_pred, target_names=['Полное', 'Неполное']))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'roc_auc': roc_auc_score(y_test, y_proba)
    }

## 6. Анализ важности признаков

In [ ]:
# Важность признаков для Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Важность')
plt.title('Топ-15 важных признаков (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nТоп-10 важных признаков:")
print(feature_importance.head(10))

## 7. ROC-кривые

In [ ]:
plt.figure(figsize=(10, 6))

for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {result['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривые моделей')
plt.legend()
plt.grid(True)
plt.show()

## 8. Матрица ошибок для лучшей модели

In [ ]:
# Выбираем модель с лучшим ROC-AUC
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_result = results[best_model_name]

print(f"Лучшая модель: {best_model_name}")

cm = confusion_matrix(y_test, best_result['y_pred'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Полное', 'Неполное'],
            yticklabels=['Полное', 'Неполное'])
plt.ylabel('Истинные значения')
plt.xlabel('Предсказанные значения')
plt.title(f'Матрица ошибок: {best_model_name}')
plt.show()

## 9. Оптимизация порога для максимизации Recall

Для минимизации потока в краудсорсинг важно максимизировать Recall при приемлемой Precision

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, best_result['y_proba'])

# Находим порог для Precision >= 0.80 и максимального Recall
target_precision = 0.80
valid_indices = precision >= target_precision
if valid_indices.any():
    best_idx = np.where(valid_indices)[0][-1]
    best_threshold = thresholds[best_idx]
    best_precision = precision[best_idx]
    best_recall = recall[best_idx]
    
    print(f"Оптимальный порог: {best_threshold:.3f}")
    print(f"Precision: {best_precision:.3f}")
    print(f"Recall: {best_recall:.3f}")
    print(f"F1-Score: {2 * best_precision * best_recall / (best_precision + best_recall):.3f}")

# Визуализация
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, label='Precision-Recall кривая')
plt.axhline(y=target_precision, color='r', linestyle='--', label=f'Target Precision = {target_precision}')
if valid_indices.any():
    plt.scatter([best_recall], [best_precision], color='red', s=100, zorder=5, label='Оптимальная точка')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall кривая')
plt.legend()
plt.grid(True)
plt.show()

## 10. Выводы и рекомендации

### Ключевые признаки неполных описаний:
1. **Количество пропущенных полей** - главный индикатор
2. **Длина описания** - короткие описания чаще неполные
3. **Количество удобств** - мало удобств = неполное описание
4. **Наличие ключевых полей** (тип кровати, вид, площадь)

### Бизнес-рекомендации:
- Внедрить модель для автоматической фильтрации неполных описаний
- Запросить у операторов дополнение данных до отправки в краудсорсинг
- Создать чек-лист обязательных полей для операторов
- Мониторить качество данных от каждого оператора

### Ожидаемый эффект:
- Сокращение потока в краудсорсинг на 20-30%
- Снижение операционных издержек
- Улучшение качества матчинга комнат

In [ ]:
# Сохранение лучшей модели
import pickle

with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_result['model'], f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Модель и scaler сохранены")